In [1]:
import os
import subprocess
import sys
import glob
import pandas as pd
from pathlib import Path
import src

/home/luisass/Downloads/tmpppp/MRE-sequence-inspection/src/mre_metadata.py:6: UserWarning: The DICOM readers are highly experimental, unstable, and only work for Siemens time-series at the moment
Please use with caution.  We would be grateful for your help in improving them
  from nibabel.nicom import csareader


## Heudiconv wrapper script MRE
This script runs heudiconv on dicom files, generating an overview tsv file that lists metadata for each image type (e.g. sequence that was used to generate it). <br>
Specify in / output paths for summary script in the config .json file in this folder (one block per scanner under `SCANNERS`). <br>

How to: <br>
usually, there is no need to change anything in this script. The only task you have is possibly **renaming some folders on the drive, such that they match the dicom patient id** - this will avoid confusion or double names for subjects. You will see mismatching names in the output dataframes below. This applies unless theres a naming conflict, that makes it impossible to use the patient id as foldername, more on this below. 

Requirements (already set in config file):
- *dicom root paths* - where to look for dicoms (per scanner). To be specified **relative**, i.e. excluding the drive, as we might mount the harddrive differently each time.
- *output paths* - where to write the summary sheets to (per scanner). To be specified **relative**, i.e. excluding the drive, as we might mount the harddrive differently each time.
- *folder templates* - how to find the dicom files (per scanner). The pattern must have the placeholders for the respective dicom root path, and the subject, more on this below.

Output:
- *.heudiconv* folder containing the summaries, one for each session. Most importantly: containing the dicominfo.tsv file with the sequence infos, or a modification version thereof (per scanner)

NB! 
- You need to have **the same** folder hierarchy (as specified by `FOLDER_TEMPLATE` in the config) for all subjects of one scanner. For subjects with a different hierarchy heudiconv will process 0 files.

---
If running outside of harddrive, make use of "root_overwrite" argument to bypass automatic detection of root

---

In [2]:
def run_heudiconv_noconvert_for_scanner(output_dir, pattern_template, sessions, custom_heuristic = None, force_heudiconv=False, dryrun = False):
    """
    Run Heudiconv without heuristic to create overview of DICOM files matching the given pattern template.

    Args:
        output_dir (str): Directory where overview sheets should be saved.
        pattern_template (str): path pattern to dicom files, must have {subject} placeholder. Example: 'A:\\MR_Data\\Raw\\7T_Terra\\{subject}\\SCANS\\*\\DICOM\\*.dcm'
        sessions (list): List of session/subject identifiers to process. This will be replacing the subject placeholders in the pattern.
        force_heudiconv (bool): Whether to re-run even if outputs exist.
        custom_heuristic (str): Path to a custom heuristic file. If None, the default heuristic will be used (provided by heudiconv, called "convertall.py").
        dryrun (bool): If True, print commands without executing them.
        
    Raises:
        ValueError: If '{subject}' is not found in the pattern_template.
        
    """
        
    if "{subject}" not in pattern_template:
        raise ValueError("The given template does not contain the {subject} placeholder. Please specify with placeholder.")
    
    if custom_heuristic:
        if not os.path.isfile(custom_heuristic):
            raise ValueError(f"The specified heuristic file does not exist: {custom_heuristic}")
        heuristic = custom_heuristic
    else:
        heuristic = "convertall"

    os.makedirs(output_dir, exist_ok=True)

    # Find previously processed sessions
    dicominfo_file_paths = glob.glob(
        r'**/dicominfo*.tsv',
        root_dir=output_dir,
        recursive=True,
        include_hidden=True
    )
    already_processed_sessions = [
        os.path.basename(os.path.dirname(os.path.dirname(p)))
        for p in dicominfo_file_paths
    ]

    print(f"Starting processing... \n")

    for i, session in enumerate(sessions):
       
        if session in already_processed_sessions and not force_heudiconv:
            print(f"=== SESSION ({i+1}/{len(sessions)}): {session:>35s} --> Already processed. Skipping.")
            continue
        else:
            print(f"==== SESSION ({i+1}/{len(sessions)}): {session:>35s} --> RUNNING HEUDICONV!")
            cmd = rf'heudiconv -d {pattern_template} -s {session} -o {output_dir} -c none -f {heuristic}'

            print(f"Command: {cmd}")
            if not dryrun:
                process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, shell=True)

                # Stream stdout in real-time
                while True:
                    output = process.stdout.readline()
                    if output == "" and process.poll() is not None:
                        break
                    if output:
                        print(output.strip())
                        sys.stdout.flush()

                # Print any remaining stderr
                stderr_output = process.stderr.read()
                if stderr_output:
                    print(stderr_output.strip())
                    sys.stdout.flush()
            else:
                print("DRYRUN - don't run heudiconv (change flag to disable).")
            

    print(f"\n... success, all sessions processed!")


In [ ]:
# Load config, all paths are resolved relative to the drive root (see config_file_comments.txt)
cfg = src.config.load_config("CONFIG_FILE.json")

#if not using from the harddrive, pass your dicom root as root_overwrite and set relative paths to '' in the config file.
#cfg = src.config.load_config("CONFIG_FILE.json", root_overwrite="/your/dicom/root")

cfg.show()

ValueError: Generated combined path '/home/luisass/Downloads/MR_Data/Raw/MRE/7T_Terra' does not exist. Segments: ('/home/luisass/Downloads', 'MR_Data/Raw/MRE/7T_Terra')

In [ ]:
#__MACOSX is a temp file from apple
IGNORE_FOLDERS = ["__MACOSX"] #ignore certain folders
IGNORE_HIDDEN = True #ignore hidden folders starting with a '.'. E.g. path/to/.thisisafoldertoignore

scanner_7T = cfg.scanners["7T"]
scanner_3T = cfg.scanners["3T"]

#scanner label -> dataframe with foldername and dicom patient_name, filled per scanner below
subjects = dict()

ScannerPaths(label='3T', dicom_root='/home/luisass/Downloads', output_root='/home/luisass/Downloads', folder_template='/home/luisass/Downloads/{subject}/*/*/*.[dI][cM][mA]')

# 7T Data

In [ ]:
#not necessary to run this cell, just for info on the heudiconv tool :)
#!heudiconv --help


Naming conflict for: 

...176_3, ...176_5, ...387_2, ...176_3 --> same subject id, hence keep the suffix <br>
Phantom --> just adding date to the name so it is not confusing <br>
invivo_fmre_...\_..._... --> underscore necessary

In [ ]:
#NB! apart from exceptions where ambiguity forces it or chars are used that cannot be part of a foldername, please make sure to rename other folders such that they match the patient_name

subjects["7T"] = src.utils.extract_subject_foldername_and_patientid(pattern_template=scanner_7T.folder_template,
                                    folders_to_ignore=IGNORE_FOLDERS,
                                    ignore_hidden=IGNORE_HIDDEN)
subjects["7T"]["patient_name"] = subjects["7T"]["patient_name"].apply(lambda x: x.replace("^", ""))
subjects["7T"]["is match?"] = subjects["7T"].foldername == subjects["7T"].patient_name


print("==========================================================")
print(f"Please rename folder according to DICOM patient_name.\
      \nNaming conflicts are an exception, please note them here.")
print("==========================================================")

subjects["7T"]

In [ ]:
sessions_7T = subjects["7T"].foldername.values.tolist()

run_heudiconv_noconvert_for_scanner(
    output_dir=scanner_7T.output_root,
    pattern_template=scanner_7T.folder_template,
    sessions=sessions_7T,
    force_heudiconv=False,
    custom_heuristic="heuristics/convertall_enhancedcm_mod.py", #use modified heuristic to extract TR and TE from enhanced DICOMs (new XB10 platform writes out enhanced format, which stores TR/TE in a different way)
    dryrun=True)

# 3T data

Naming conflict for: 

MRE_test_.../.../... --> no slashes allowed in foldername, using underscores

In [7]:
subjects["3T"] = src.utils.extract_subject_foldername_and_patientid(pattern_template=scanner_3T.folder_template,
                                    folders_to_ignore=IGNORE_FOLDERS,
                                    ignore_hidden=IGNORE_HIDDEN)
subjects["3T"]["is match?"] = subjects["3T"].foldername == subjects["3T"].patient_name

print("==========================================================")
print(f"Please rename folder according to DICOM patient_name.\
      \nNaming conflicts are an exception, please note them here.")
print("==========================================================")

subjects["3T"]

Skipping folder 'matlab_R2025b_Linux': No match found for glob query '/home/luisass/Downloads/matlab_R2025b_Linux/*/*/*.[dI][cM][mA]' with current template. Check folder hierarchy.
Skipping folder 'DPIA_MH_TGA_project.docx': No match found for glob query '/home/luisass/Downloads/DPIA_MH_TGA_project.docx/*/*/*.[dI][cM][mA]' with current template. Check folder hierarchy.
Skipping folder 'Cabral_et_al_2023_intrinsic_macroscale_oscillatory_modes': No match found for glob query '/home/luisass/Downloads/Cabral_et_al_2023_intrinsic_macroscale_oscillatory_modes/*/*/*.[dI][cM][mA]' with current template. Check folder hierarchy.
Skipping folder 'Scan_Summaries.zip': No match found for glob query '/home/luisass/Downloads/Scan_Summaries.zip/*/*/*.[dI][cM][mA]' with current template. Check folder hierarchy.
Skipping folder 'Data_V3.zip.crdownload': No match found for glob query '/home/luisass/Downloads/Data_V3.zip.crdownload/*/*/*.[dI][cM][mA]' with current template. Check folder hierarchy.
Skippin

,foldername,patient_name,is match?
0,Phantom^Bottle_Phantom_Bottle_19900101,Phantom^Bottle,False


In [ ]:
sessions_3T = subjects["3T"].foldername.values.tolist()

run_heudiconv_noconvert_for_scanner(
    output_dir=scanner_3T.output_root,
    pattern_template=scanner_3T.folder_template,
    sessions=sessions_3T,
    custom_heuristic="heuristics/convertall_enhancedcm_mod.py", #use modified heuristic to extract TR and TE from enhanced DICOMs (new XB10 platform writes out enhanced format, which stores TR/TE in a different way)
    force_heudiconv=False,
    dryrun=False)

# Delete sensitive columns and move TR / TE values for enhanced format dicoms

- we do not want e.g. patient age or sex in the overview sheets, thus delete! Overview sheets will be copied without named columns and renamed to specified TSV name.

- the enhanced format leads to failure of heudiconv to extrat TR / TE. We have manually exctracted them and saved in the custom column. <br>
Copy them to the TR and TE column manually.

In [ ]:
dicominfo_file_paths = []
for scanner in cfg.scanners.values():
    cur_dicominfo_file_paths = glob.glob(r'**/dicominfo*.tsv', root_dir= scanner.output_root, recursive= True, include_hidden=True)
    print(f"{len(cur_dicominfo_file_paths)} dicominfo files in {scanner.output_root}")
    dicominfo_file_paths.extend(os.path.join(scanner.output_root, p) for p in cur_dicominfo_file_paths)

print()
#to read out scan IDs of dicominfos (just for sanity check next cell)
dicominfo_scan_IDs = list()

columns_to_delete = ["patient_age", "patient_sex"]
for info_file_path in dicominfo_file_paths:
    if os.path.exists(info_file_path):
        subject_name = os.path.basename(os.path.dirname(os.path.dirname(info_file_path)))
        dicominfo_scan_IDs.append(subject_name)
        tsv_file = pd.read_csv(info_file_path, sep='\t', encoding='latin-1', on_bad_lines='skip')

        print(f"[{subject_name}]")

        # overwrite TR/TE from custom column for enhanced DICOMs
        if "custom" in tsv_file.columns:
            def extract_custom(custom_val, key):
                try:
                    return dict(eval(custom_val)).get(key)
                except Exception:
                    return None
                
            extracted_values = []
            for idx, row in tsv_file.iterrows():
                custom_tr = extract_custom(row["custom"], "TR")
                custom_te = extract_custom(row["custom"], "TE")

                if custom_tr is not None and row["TR"] == -1:
                    tsv_file.at[idx, "TR"] = custom_tr
                    extracted_values.append("TR")

                if custom_te is not None and row["TE"] == -1:
                    tsv_file.at[idx, "TE"] = custom_te
                    extracted_values.append("TE")

            if extracted_values:
                print(f"  enhanced DICOM  →  TR extracted: {extracted_values.count('TR')}x  |  TE extracted: {extracted_values.count('TE')}x")

        # Check if any of the columns to delete exist in the DataFrame
        if any(c in tsv_file.columns for c in columns_to_delete):
            tsv_file.drop(columns=columns_to_delete, inplace=True, errors="ignore")
            cur_dicominfo_filename = cfg.dicominfo_tsv_name.format(subject=subject_name)
            new_file_path = os.path.join(os.path.dirname(info_file_path), cur_dicominfo_filename)
            tsv_file.to_csv(new_file_path, sep='\t', index=False)
            print(f"  sensitive cols  →  removed {columns_to_delete}")
            os.remove(info_file_path)
        else:
            print(f"  sensitive cols  →  none found, skipping")

        print()
        

# Check for missing data or missing info sheet
do we have info sheets for all data, and data for all info sheets?

In [ ]:
#folder names of all scanners whose section was run above
foldernames = [name for scanner_subjects in subjects.values() for name in scanner_subjects.foldername]
if sorted(foldernames) != sorted(dicominfo_scan_IDs):
    no_info = list(set(foldernames) - set(dicominfo_scan_IDs))
    no_data = list(set(dicominfo_scan_IDs) - set(foldernames))
    print(f"Existing scans that have no dicominfo.tsv: {no_info}")
    print(f"Existing dicominfo.tsv where no scan is found: {no_data}")
    raise ValueError("Found mismatch in found Scan data and existing summary sheets. See print output above.")
else:
    print("For all scans a dicominfo sheet was found, and for all dicominfo sheets a scan was found.")